# Advanced Problems With Solutions — Deleting Properties

This notebook revisits **deleting properties** in Python in a tutorial style.

Each problem is broken into small logical steps. We will first discuss the design, then implement one piece at a time, inspect the behavior, handle edge cases, and finish with a complete solution or test.

The focus is not only on syntax. The deeper goal is to understand what deletion should **mean** for an object.

## Core Idea

When a class defines a property deleter, code such as:

```python
del obj.name
```

does not remove the property from the class. Python invokes the property's deleter for that instance.

The property object remains on the class; only the instance state changes.

A deleter can implement many policies:

- remove backing state,
- restore a default,
- invalidate caches,
- archive an old value,
- close a resource,
- refuse deletion,
- enforce an invariant,
- or record an audit event.

# Problem 1 — Delete and Recreate a Property Value

Create a `Member` whose `screen_name` can be deleted and later assigned again. After deletion, reading it should raise `AttributeError`.

## Step 1 — Separate Public API From Storage

The public attribute will be `screen_name`, while the internal value will live in `_screen_name`. Using a different backing name prevents recursive property access.

In [1]:
class Member:
    def __init__(self, screen_name):
        self.screen_name = screen_name

    @property
    def screen_name(self):
        return self._screen_name

    @screen_name.setter
    def screen_name(self, value):
        if not isinstance(value, str):
            raise TypeError("screen_name must be a string")
        value = value.strip()
        if not value:
            raise ValueError("screen_name cannot be empty")
        self._screen_name = value

## Step 2 — Add the Deleter

The simplest deleter removes the backing attribute. The property itself stays on the class.

In [2]:
class Member:
    def __init__(self, screen_name):
        self.screen_name = screen_name

    @property
    def screen_name(self):
        return self._screen_name

    @screen_name.setter
    def screen_name(self, value):
        if not isinstance(value, str):
            raise TypeError("screen_name must be a string")
        value = value.strip()
        if not value:
            raise ValueError("screen_name cannot be empty")
        self._screen_name = value

    @screen_name.deleter
    def screen_name(self):
        del self._screen_name

## Step 3 — Inspect Before and After Deletion

In [3]:
m = Member("Ada")
print("before:", m.__dict__)
del m.screen_name
print("after:", m.__dict__)
print("class still has property:", isinstance(Member.screen_name, property))

before: {'_screen_name': 'Ada'}
after: {}
class still has property: True


## Step 4 — Recreate the Value

The setter still exists, so the deleted instance value can be created again.

In [4]:
try:
    print(m.screen_name)
except AttributeError as ex:
    print("expected:", ex)

m.screen_name = "Grace"
print(m.screen_name)

expected: 'Member' object has no attribute '_screen_name'
Grace


### Takeaway

Deleting a property value changes **instance state**. It does not destroy the class-level property.

# Problem 2 — Make Deletion Idempotent

The previous deleter fails if called twice. For optional state, repeated deletion is often easier to work with if it is safe.

## Step 1 — Choose a Policy

We will define repeated deletion as a no-op. This makes the operation **idempotent**.

## Step 2 — Check for the Backing Attribute

In [5]:
class SafeMember:
    def __init__(self, screen_name):
        self.screen_name = screen_name

    @property
    def screen_name(self):
        if "_screen_name" not in self.__dict__:
            raise AttributeError("screen_name has been deleted")
        return self._screen_name

    @screen_name.setter
    def screen_name(self, value):
        value = str(value).strip()
        if not value:
            raise ValueError("screen_name cannot be empty")
        self._screen_name = value

    @screen_name.deleter
    def screen_name(self):
        if "_screen_name" in self.__dict__:
            del self._screen_name

## Step 3 — Test Repeated Deletion

In [6]:
m = SafeMember("Linus")
del m.screen_name
del m.screen_name
del m.screen_name
print(m.__dict__)

{}


## Step 4 — Compact Variant

For ordinary `__dict__`-backed objects, this is also common:

```python
self.__dict__.pop("_screen_name", None)
```

The default prevents `KeyError`.

# Problem 3 — Deletion Means Restore the Default

A `ViewSettings` object must always have a valid zoom level. Deleting `zoom_level` should restore the default of `100` rather than remove the backing attribute.

## Step 1 — Store the Default at Class Level

In [7]:
class ViewSettings:
    DEFAULT_ZOOM = 100

## Step 2 — Validate Assignment

In [8]:
class ViewSettings:
    DEFAULT_ZOOM = 100

    def __init__(self, zoom_level=DEFAULT_ZOOM):
        self.zoom_level = zoom_level

    @property
    def zoom_level(self):
        return self._zoom_level

    @zoom_level.setter
    def zoom_level(self, value):
        if not isinstance(value, int) or isinstance(value, bool):
            raise TypeError("zoom_level must be an integer")
        if not 50 <= value <= 200:
            raise ValueError("zoom_level must be between 50 and 200")
        self._zoom_level = value

## Step 3 — Implement Reset-on-Delete

A deleter does not have to use Python's `del` statement. It can implement any domain-specific reset behavior.

In [9]:
class ViewSettings:
    DEFAULT_ZOOM = 100

    def __init__(self, zoom_level=DEFAULT_ZOOM):
        self.zoom_level = zoom_level

    @property
    def zoom_level(self):
        return self._zoom_level

    @zoom_level.setter
    def zoom_level(self, value):
        if not isinstance(value, int) or isinstance(value, bool):
            raise TypeError("zoom_level must be an integer")
        if not 50 <= value <= 200:
            raise ValueError("zoom_level must be between 50 and 200")
        self._zoom_level = value

    @zoom_level.deleter
    def zoom_level(self):
        self._zoom_level = self.DEFAULT_ZOOM

settings = ViewSettings(175)
print(settings.zoom_level)
del settings.zoom_level
print(settings.zoom_level)

175
100


### Takeaway

Deletion semantics are part of your API design. Sometimes "delete" means "return to default".

# Problem 4 — Forbid Deletion Explicitly

An `Invoice` has a permanent `invoice_number`. It should be readable, not assignable through the property, and not deletable.

## Step 1 — A Getter Without a Setter Is Read-Only

Python will already reject assignment if no setter exists.

In [10]:
class Invoice:
    def __init__(self, invoice_number):
        self._invoice_number = str(invoice_number)

    @property
    def invoice_number(self):
        return self._invoice_number

## Step 2 — Add a Deleter Only to Explain the Rule

A missing deleter already blocks deletion, but an explicit deleter can provide a clearer domain error.

In [11]:
class Invoice:
    def __init__(self, invoice_number):
        value = str(invoice_number).strip()
        if not value:
            raise ValueError("invoice_number cannot be empty")
        self._invoice_number = value

    @property
    def invoice_number(self):
        return self._invoice_number

    @invoice_number.deleter
    def invoice_number(self):
        raise AttributeError("invoice_number is permanent and cannot be deleted")

invoice = Invoice("INV-1001")
try:
    del invoice.invoice_number
except AttributeError as ex:
    print(ex)

invoice_number is permanent and cannot be deleted


### Takeaway

A deleter is useful even when the correct behavior is to refuse deletion.

# Problem 5 — Invalidate a Dependent Cache

A `SearchPhrase` stores `text` and lazily caches a normalized version. If `text` changes or disappears, the cached normalized value must be cleared.

## Step 1 — Build the Source Property

In [12]:
class SearchPhrase:
    def __init__(self, text):
        self.text = text

    @property
    def text(self):
        if "_text" not in self.__dict__:
            raise AttributeError("text is not set")
        return self._text

    @text.setter
    def text(self, value):
        value = str(value).strip()
        if not value:
            raise ValueError("text cannot be empty")
        self._text = value

## Step 2 — Centralize Cache Invalidation

Both assignment and deletion need the same cleanup. A helper method avoids duplication.

In [13]:
class SearchPhrase:
    def __init__(self, text):
        self.text = text

    def _invalidate_normalized(self):
        self.__dict__.pop("_normalized_cache", None)

    @property
    def text(self):
        if "_text" not in self.__dict__:
            raise AttributeError("text is not set")
        return self._text

    @text.setter
    def text(self, value):
        value = str(value).strip()
        if not value:
            raise ValueError("text cannot be empty")
        self._text = value
        self._invalidate_normalized()

    @text.deleter
    def text(self):
        self.__dict__.pop("_text", None)
        self._invalidate_normalized()

    @property
    def normalized(self):
        if "_normalized_cache" not in self.__dict__:
            print("normalizing...")
            self._normalized_cache = " ".join(self.text.lower().split())
        return self._normalized_cache

## Step 3 — Observe Cache Behavior

In [14]:
p = SearchPhrase("  Advanced   Python  ")
print(p.normalized)
print(p.normalized)
print(p.__dict__)
del p.text
print(p.__dict__)
try:
    print(p.normalized)
except AttributeError as ex:
    print(ex)

normalizing...
advanced python
advanced python
{'_text': 'Advanced   Python', '_normalized_cache': 'advanced python'}
{}
normalizing...
text is not set


### Design Question

Whenever you delete source state, ask: **what derived state becomes stale?**

# Problem 6 — One Property Backed by Multiple Attributes

A `CanvasSize.dimensions` property is backed by `_width` and `_height`. Deleting the property should remove both pieces of state.

## Step 1 — Model the Property as One Logical Concept

The public value will be a two-item tuple `(width, height)`.

In [15]:
class CanvasSize:
    def __init__(self, width, height):
        self.dimensions = (width, height)

    @property
    def dimensions(self):
        return (self._width, self._height)

    @dimensions.setter
    def dimensions(self, value):
        try:
            width, height = value
        except (TypeError, ValueError):
            raise ValueError("dimensions must contain exactly two values") from None
        if width <= 0 or height <= 0:
            raise ValueError("both dimensions must be positive")
        self._width = width
        self._height = height

## Step 2 — Prevent Partial Logical State

If either internal attribute is missing, the logical property is unavailable.

In [16]:
class CanvasSize:
    def __init__(self, width, height):
        self.dimensions = (width, height)

    @property
    def dimensions(self):
        if "_width" not in self.__dict__ or "_height" not in self.__dict__:
            raise AttributeError("dimensions are not set")
        return (self._width, self._height)

    @dimensions.setter
    def dimensions(self, value):
        try:
            width, height = value
        except (TypeError, ValueError):
            raise ValueError("dimensions must contain exactly two values") from None
        if width <= 0 or height <= 0:
            raise ValueError("both dimensions must be positive")
        self._width = width
        self._height = height

    @dimensions.deleter
    def dimensions(self):
        self.__dict__.pop("_width", None)
        self.__dict__.pop("_height", None)

## Step 3 — Test the Multi-Attribute Deletion

In [17]:
size = CanvasSize(1920, 1080)
print(size.dimensions)
del size.dimensions
print(size.__dict__)
try:
    print(size.dimensions)
except AttributeError as ex:
    print(ex)

(1920, 1080)
{}
dimensions are not set


### Takeaway

A single property can represent a logical value stored across several internal attributes.

# Problem 7 — Distinguish Deleted From `None`

Suppose `secondary_email=None` is a valid explicit value. We also need a distinct state meaning "deleted or missing". A sentinel solves that ambiguity.

## Step 1 — Create a Unique Sentinel

In [18]:
MISSING = object()

## Step 2 — Store the Sentinel on Deletion

In [19]:
class ContactRecord:
    def __init__(self, secondary_email=None):
        self._secondary_email = secondary_email

    @property
    def secondary_email(self):
        if self._secondary_email is MISSING:
            raise AttributeError("secondary_email is missing")
        return self._secondary_email

    @secondary_email.setter
    def secondary_email(self, value):
        if value is not None and not isinstance(value, str):
            raise TypeError("secondary_email must be str or None")
        self._secondary_email = value

    @secondary_email.deleter
    def secondary_email(self):
        self._secondary_email = MISSING

## Step 3 — Compare the States

In [20]:
r = ContactRecord(None)
print("explicit None:", r.secondary_email)
del r.secondary_email
try:
    print(r.secondary_email)
except AttributeError as ex:
    print("deleted:", ex)
r.secondary_email = None
print("explicit None again:", r.secondary_email)

explicit None: None
deleted: secondary_email is missing
explicit None again: None


### Takeaway

Use a sentinel when `None`, `0`, `False`, or another ordinary value must remain distinguishable from a missing state.

# Problem 8 — Deletion Depends on Object State

A `BlogPost.slug` may be changed or deleted while the post is a draft. After publication, it becomes permanent.

## Step 1 — Add an Explicit State

In [21]:
class BlogPost:
    def __init__(self, slug):
        self.status = "draft"
        self.slug = slug

## Step 2 — Make Both Setter and Deleter State-Aware

In [22]:
class BlogPost:
    def __init__(self, slug):
        self.status = "draft"
        self.slug = slug

    @property
    def slug(self):
        if "_slug" not in self.__dict__:
            raise AttributeError("slug is not set")
        return self._slug

    @slug.setter
    def slug(self, value):
        if self.status != "draft":
            raise RuntimeError("slug cannot change after publication")
        value = str(value).strip()
        if not value:
            raise ValueError("slug cannot be empty")
        self._slug = value

    @slug.deleter
    def slug(self):
        if self.status != "draft":
            raise RuntimeError("slug cannot be deleted after publication")
        self.__dict__.pop("_slug", None)

    def publish(self):
        if "_slug" not in self.__dict__:
            raise RuntimeError("cannot publish without a slug")
        self.status = "published"

## Step 3 — Test Legal and Illegal Deletion

In [23]:
post = BlogPost("property-deletion")
del post.slug
post.slug = "advanced-properties"
post.publish()
try:
    del post.slug
except RuntimeError as ex:
    print(ex)

slug cannot be deleted after publication


### Design Lesson

Sometimes the real question is not "can this property be deleted?" but "can it be deleted **right now**?"

# Problem 9 — Extend a Base-Class Deleter

A base class already implements correct deletion behavior. A subclass wants to add counting without duplicating the base logic.

## Step 1 — Define the Base Property

In [24]:
class NamedItem:
    def __init__(self, label):
        self.label = label

    @property
    def label(self):
        if "_label" not in self.__dict__:
            raise AttributeError("label is not set")
        return self._label

    @label.setter
    def label(self, value):
        value = str(value).strip()
        if not value:
            raise ValueError("label cannot be empty")
        self._label = value

    @label.deleter
    def label(self):
        self.__dict__.pop("_label", None)

## Step 2 — Inspect the Property

A property exposes its underlying functions through `fget`, `fset`, and `fdel`.

In [25]:
print(NamedItem.label.fget)
print(NamedItem.label.fset)
print(NamedItem.label.fdel)

<function NamedItem.label at 0x0000020C7A992C00>
<function NamedItem.label at 0x0000020C7A992CA0>
<function NamedItem.label at 0x0000020C7A992D40>


## Step 3 — Extend the Deleter in a Subclass

In [26]:
class CountedNamedItem(NamedItem):
    def __init__(self, label):
        self.delete_count = 0
        super().__init__(label)

    @NamedItem.label.deleter
    def label(self):
        self.delete_count += 1
        NamedItem.label.fdel(self)

item = CountedNamedItem("demo")
del item.label
del item.label
print(item.delete_count)
print(item.__dict__)

2
{'delete_count': 2}


### Takeaway

Reusing the base `fdel` is one way to extend deletion semantics without rewriting them.

# Problem 10 — Property Deletion With `__slots__`

Classes using `__slots__` may not have `__dict__`. A deletion strategy that relies on `self.__dict__.pop(...)` is therefore not universal.

## Step 1 — Use Attribute Operations Instead

In [27]:
class SlottedCode:
    __slots__ = ("_code",)

    def __init__(self, code):
        self.code = code

    @property
    def code(self):
        try:
            return self._code
        except AttributeError:
            raise AttributeError("code has been deleted") from None

    @code.setter
    def code(self, value):
        value = str(value).strip()
        if not value:
            raise ValueError("code cannot be empty")
        self._code = value

    @code.deleter
    def code(self):
        if hasattr(self, "_code"):
            del self._code

## Step 2 — Test Repeated Deletion

In [28]:
s = SlottedCode("ABC")
print(s.code)
del s.code
del s.code
try:
    print(s.code)
except AttributeError as ex:
    print(ex)

ABC
code has been deleted


### Takeaway

Choose deletion logic that matches the object's storage model.

# Problem 11 — Archive Before Deleting

A `Promotion` has an active `campaign_code`. Deleting it should remove the active value but preserve the old value in history.

## Step 1 — Keep Active and Historical State Separate

In [29]:
class Promotion:
    def __init__(self, campaign_code=None):
        self._code_history = []
        if campaign_code is not None:
            self.campaign_code = campaign_code

## Step 2 — Archive the Value in the Deleter

In [30]:
class Promotion:
    def __init__(self, campaign_code=None):
        self._code_history = []
        if campaign_code is not None:
            self.campaign_code = campaign_code

    @property
    def campaign_code(self):
        if "_campaign_code" not in self.__dict__:
            raise AttributeError("no active campaign code")
        return self._campaign_code

    @campaign_code.setter
    def campaign_code(self, value):
        value = str(value).strip().upper()
        if not value:
            raise ValueError("campaign_code cannot be empty")
        self._campaign_code = value

    @campaign_code.deleter
    def campaign_code(self):
        if "_campaign_code" in self.__dict__:
            self._code_history.append(self._campaign_code)
            del self._campaign_code

    @property
    def code_history(self):
        return tuple(self._code_history)

## Step 3 — Test the Archive

In [31]:
p = Promotion("launch25")
print(p.campaign_code)
del p.campaign_code
print(p.code_history)
p.campaign_code = "winter10"
print(p.campaign_code)
print(p.code_history)

LAUNCH25
('LAUNCH25',)
WINTER10
('LAUNCH25',)


### Takeaway

Deletion can remove the active value without destroying historical information.

# Problem 12 — Delete a Resource and Perform Cleanup

A property may own a resource. Deleting the property should close the resource and remove the stored reference.

## Step 1 — Create a Fake Resource

Using a fake object keeps the notebook self-contained and deterministic.

In [32]:
class FakeHandle:
    def __init__(self):
        self.closed = False
        print("handle opened")

    def close(self):
        if not self.closed:
            self.closed = True
            print("handle closed")

## Step 2 — Create the Resource Lazily and Clean It Up on Delete

In [33]:
class HandleOwner:
    @property
    def handle(self):
        if "_handle" not in self.__dict__:
            self._handle = FakeHandle()
        return self._handle

    @handle.deleter
    def handle(self):
        handle = self.__dict__.pop("_handle", None)
        if handle is not None:
            handle.close()

## Step 3 — Test Cleanup and Idempotence

In [34]:
owner = HandleOwner()
h = owner.handle
del owner.handle
print("closed:", h.closed)
print(owner.__dict__)
del owner.handle

handle opened
handle closed
closed: True
{}


### Best-Practice Note

For short-lived resources, context managers are often preferable. Property deletion is still useful when the object deliberately owns a longer-lived optional resource.

# Problem 13 — Preserve an Invariant During Deletion

A `RecoveryProfile` may have a recovery email, a recovery phone, or both. At least one recovery method must always remain.

## Step 1 — Count the Available Recovery Methods

In [35]:
class RecoveryProfile:
    def __init__(self, email=None, phone=None):
        if email is None and phone is None:
            raise ValueError("at least one recovery method is required")
        if email is not None:
            self.email = email
        if phone is not None:
            self.phone = phone

    def _method_count(self):
        return int("_email" in self.__dict__) + int("_phone" in self.__dict__)

## Step 2 — Refuse to Delete the Final Method

In [36]:
class RecoveryProfile:
    def __init__(self, email=None, phone=None):
        if email is None and phone is None:
            raise ValueError("at least one recovery method is required")
        if email is not None:
            self.email = email
        if phone is not None:
            self.phone = phone

    def _method_count(self):
        return int("_email" in self.__dict__) + int("_phone" in self.__dict__)

    @property
    def email(self):
        if "_email" not in self.__dict__:
            raise AttributeError("email is not set")
        return self._email

    @email.setter
    def email(self, value):
        value = str(value).strip()
        if "@" not in value:
            raise ValueError("invalid email")
        self._email = value

    @email.deleter
    def email(self):
        if "_email" not in self.__dict__:
            return
        if self._method_count() == 1:
            raise RuntimeError("cannot delete the final recovery method")
        del self._email

    @property
    def phone(self):
        if "_phone" not in self.__dict__:
            raise AttributeError("phone is not set")
        return self._phone

    @phone.setter
    def phone(self, value):
        value = str(value).strip()
        if not value:
            raise ValueError("phone cannot be empty")
        self._phone = value

    @phone.deleter
    def phone(self):
        if "_phone" not in self.__dict__:
            return
        if self._method_count() == 1:
            raise RuntimeError("cannot delete the final recovery method")
        del self._phone

## Step 3 — Test Legal and Illegal Deletion

In [37]:
rp = RecoveryProfile("recover@example.com", "+359000000")
del rp.email
print(rp.__dict__)
try:
    del rp.phone
except RuntimeError as ex:
    print(ex)

{'_phone': '+359000000'}
cannot delete the final recovery method


### Takeaway

A good deleter protects the validity of the object, not just its attributes.

# Problem 14 — Reuse Deletion Behavior With a Descriptor

When several fields need the same validation and deletion behavior, a custom descriptor can be clearer than many nearly identical properties.

## Step 1 — Descriptor Hooks

A reusable descriptor can define `__set_name__`, `__get__`, `__set__`, and `__delete__`. The last one is the reusable equivalent of a property deleter.

## Step 2 — Implement the Descriptor

In [38]:
class RequiredText:
    def __set_name__(self, owner, name):
        self.public_name = name
        self.private_name = f"_{name}"

    def __get__(self, instance, owner):
        if instance is None:
            return self
        if not hasattr(instance, self.private_name):
            raise AttributeError(f"{self.public_name} has been deleted")
        return getattr(instance, self.private_name)

    def __set__(self, instance, value):
        if not isinstance(value, str):
            raise TypeError(f"{self.public_name} must be a string")
        value = value.strip()
        if not value:
            raise ValueError(f"{self.public_name} cannot be empty")
        setattr(instance, self.private_name, value)

    def __delete__(self, instance):
        if hasattr(instance, self.private_name):
            delattr(instance, self.private_name)

## Step 3 — Reuse It Across Multiple Fields

In [39]:
class BookMetadata:
    title = RequiredText()
    author = RequiredText()
    category = RequiredText()

    def __init__(self, title, author, category):
        self.title = title
        self.author = author
        self.category = category

book = BookMetadata("Python Internals", "A. Developer", "Programming")
print(book.__dict__)
del book.category
print(book.__dict__)

{'_title': 'Python Internals', '_author': 'A. Developer', '_category': 'Programming'}
{'_title': 'Python Internals', '_author': 'A. Developer'}


### Takeaway

Use descriptors when the same get/set/delete policy appears repeatedly across fields or classes.

# Problem 15 — Read-Only Generated Property That Can Be Reset

A `RequestInfo.request_id` is generated lazily. Callers cannot assign it manually, but deleting it should force generation of a new value next time it is read.

## Step 1 — Build a Getter With No Setter

In [40]:
from uuid import uuid4

class RequestInfo:
    @property
    def request_id(self):
        if "_request_id" not in self.__dict__:
            self._request_id = uuid4().hex
        return self._request_id

## Step 2 — Add a Deleter That Clears the Generated Value

In [41]:
class RequestInfo:
    @property
    def request_id(self):
        if "_request_id" not in self.__dict__:
            self._request_id = uuid4().hex
        return self._request_id

    @request_id.deleter
    def request_id(self):
        self.__dict__.pop("_request_id", None)

## Step 3 — Confirm Regeneration

In [42]:
info = RequestInfo()
first = info.request_id
del info.request_id
second = info.request_id
print(first)
print(second)
print("changed:", first != second)

1237d66b95fe43cc88b43815816c2629
79ae3d5a3c2b40eca20a57d9f44cf1cd
changed: True


### Takeaway

A property can be read-only for assignment yet still support deletion as a reset operation.

# Problem 16 — Test Deletion Behavior Systematically

Property deletion is part of your public API, so it deserves explicit tests.

We will test a small `ResettablePin` class. The important cases are: normal read, delete, read-after-delete, repeated delete, and reassignment.

In [43]:
class ResettablePin:
    def __init__(self, pin):
        self.pin = pin

    @property
    def pin(self):
        if "_pin" not in self.__dict__:
            raise AttributeError("pin has been deleted")
        return self._pin

    @pin.setter
    def pin(self, value):
        value = str(value)
        if not value.isdigit() or len(value) != 4:
            raise ValueError("pin must contain exactly 4 digits")
        self._pin = value

    @pin.deleter
    def pin(self):
        self.__dict__.pop("_pin", None)

## Step 1 — Simple Assertions

In [44]:
pin = ResettablePin("1234")
assert pin.pin == "1234"
del pin.pin
assert "_pin" not in pin.__dict__
try:
    pin.pin
except AttributeError:
    pass
else:
    raise AssertionError("Expected AttributeError")
del pin.pin
pin.pin = "5678"
assert pin.pin == "5678"
print("simple assertions passed")

simple assertions passed


## Step 2 — `unittest` Version

In [45]:
import unittest

class TestResettablePin(unittest.TestCase):
    def test_delete_removes_storage(self):
        obj = ResettablePin("1234")
        del obj.pin
        self.assertNotIn("_pin", obj.__dict__)

    def test_read_after_delete_raises(self):
        obj = ResettablePin("1234")
        del obj.pin
        with self.assertRaises(AttributeError):
            _ = obj.pin

    def test_repeated_delete_is_safe(self):
        obj = ResettablePin("1234")
        del obj.pin
        del obj.pin
        self.assertNotIn("_pin", obj.__dict__)

    def test_reassignment_after_delete(self):
        obj = ResettablePin("1234")
        del obj.pin
        obj.pin = "9876"
        self.assertEqual(obj.pin, "9876")

suite = unittest.defaultTestLoader.loadTestsFromTestCase(TestResettablePin)
result = unittest.TextTestRunner(verbosity=2).run(suite)
print("successful:", result.wasSuccessful())

test_delete_removes_storage (__main__.TestResettablePin.test_delete_removes_storage) ... ok
test_read_after_delete_raises (__main__.TestResettablePin.test_read_after_delete_raises) ... ok
test_reassignment_after_delete (__main__.TestResettablePin.test_reassignment_after_delete) ... ok
test_repeated_delete_is_safe (__main__.TestResettablePin.test_repeated_delete_is_safe) ... ok

----------------------------------------------------------------------
Ran 4 tests in 0.009s

OK


successful: True


# Final Integrated Problem — Export Configuration

Build an `ExportConfig` that combines several deletion policies.

### `destination`
- required,
- validated,
- cannot be deleted.

### `access_key`
- optional,
- deletion is idempotent,
- reading after deletion raises `AttributeError`,
- changing or deleting it invalidates cached headers.

### `batch_size`
- positive integer,
- deleting it restores a default.

### `headers`
- lazily computed and cached.

### Audit
- every access-key deletion attempt is recorded.

## Step 1 — Identify the State

We need `_destination`, optional `_access_key`, `_batch_size`, optional `_headers_cache`, and `audit_log`.

## Step 2 — Separate the Three Deletion Policies

- `destination`: deletion is forbidden.
- `access_key`: deletion removes optional state and clears dependent cache.
- `batch_size`: deletion restores a default.

## Step 3 — Complete Solution

In [46]:
from datetime import datetime, timezone

class ExportConfig:
    DEFAULT_BATCH_SIZE = 500

    def __init__(self, destination, access_key=None, batch_size=DEFAULT_BATCH_SIZE):
        self.audit_log = []
        self.destination = destination
        self.batch_size = batch_size
        if access_key is not None:
            self.access_key = access_key

    @property
    def destination(self):
        return self._destination

    @destination.setter
    def destination(self, value):
        value = str(value).strip()
        if not value:
            raise ValueError("destination cannot be empty")
        self._destination = value

    @destination.deleter
    def destination(self):
        raise AttributeError("destination is required and cannot be deleted")

    def _invalidate_headers(self):
        self.__dict__.pop("_headers_cache", None)

    @property
    def access_key(self):
        if "_access_key" not in self.__dict__:
            raise AttributeError("access_key is not configured")
        return self._access_key

    @access_key.setter
    def access_key(self, value):
        if not isinstance(value, str):
            raise TypeError("access_key must be a string")
        value = value.strip()
        if len(value) < 8:
            raise ValueError("access_key must contain at least 8 characters")
        self._access_key = value
        self._invalidate_headers()

    @access_key.deleter
    def access_key(self):
        existed = "_access_key" in self.__dict__
        self.__dict__.pop("_access_key", None)
        if existed:
            self._invalidate_headers()
        self.audit_log.append({
            "event": "access_key_deleted",
            "existed_before": existed,
            "timestamp": datetime.now(timezone.utc).isoformat(),
        })

    @property
    def batch_size(self):
        return self._batch_size

    @batch_size.setter
    def batch_size(self, value):
        if not isinstance(value, int) or isinstance(value, bool):
            raise TypeError("batch_size must be an integer")
        if value <= 0:
            raise ValueError("batch_size must be positive")
        self._batch_size = value

    @batch_size.deleter
    def batch_size(self):
        self._batch_size = self.DEFAULT_BATCH_SIZE

    @property
    def headers(self):
        if "_headers_cache" not in self.__dict__:
            headers = {"Accept": "application/json"}
            if "_access_key" in self.__dict__:
                headers["Authorization"] = f"Bearer {self._access_key}"
            self._headers_cache = headers
        return dict(self._headers_cache)

## Step 4 — Exercise the Different Behaviors

In [47]:
config = ExportConfig(
    destination="/tmp/export.json",
    access_key="secret-key-123",
    batch_size=1000,
)

print(config.headers)
del config.access_key
print(config.headers)
print(config.audit_log[-1])

del config.access_key
print(config.audit_log[-1])

print("batch before:", config.batch_size)
del config.batch_size
print("batch after:", config.batch_size)

try:
    del config.destination
except AttributeError as ex:
    print(ex)

{'Accept': 'application/json', 'Authorization': 'Bearer secret-key-123'}
{'Accept': 'application/json'}
{'event': 'access_key_deleted', 'existed_before': True, 'timestamp': '2026-09-18T14:20:16.366932+00:00'}
{'event': 'access_key_deleted', 'existed_before': False, 'timestamp': '2026-09-18T14:20:16.367168+00:00'}
batch before: 1000
batch after: 500
destination is required and cannot be deleted


### What This Final Example Combines

- forbidden deletion,
- optional-state deletion,
- idempotence,
- reset-on-delete,
- cache invalidation,
- lazy computation,
- audit logging.

# Common Mistakes

## Mistake 1 — Bypassing the Public Deleter

`del obj._value` bypasses all logic in `@value.deleter`. If the deleter performs cleanup, auditing, or validation, that behavior is skipped.

## Mistake 2 — Recursive Deletion

Inside a deleter, `del self.value` usually calls the same deleter again. Delete or reset the backing state instead.

## Mistake 3 — Catching Every Exception

Avoid `except Exception: pass`. It can hide unrelated bugs. Prefer a precise existence check or a precise exception.

## Mistake 4 — Forgetting Dependent State

If a cache or derived property depends on the deleted value, invalidate it too.

## Mistake 5 — Assuming `__dict__` Always Exists

Slotted classes may not have an instance dictionary. Match deletion logic to the storage model.

## Mistake 6 — Using `None` for Two Meanings

If `None` is a valid value, use a sentinel for the deleted/missing state.

# Best-Practice Checklist

When designing a deletable property, ask:

1. What does deletion mean in this domain?
2. Should repeated deletion be safe?
3. What should reading after deletion do?
4. Does anything depend on the deleted value?
5. Can deletion violate an invariant?
6. Should deletion restore a default instead?
7. Does deletion need cleanup or auditing?
8. Does the object use `__dict__` or `__slots__`?
9. Would a reusable descriptor be clearer?
10. Have deletion and reassignment both been tested?

# Additional Practice Problems

Try these without looking back at the solutions:

1. **Sort override:** deleting `order` restores the default sort order.
2. **Filename cache:** deleting `filename` also invalidates a cached extension.
3. **Protected owner:** `owner` may be deleted only while `status == "draft"`.
4. **Archived note:** deleting `note` moves it into `history`.
5. **Slotted nickname:** implement get/set/repeated-delete using `__slots__`.
6. **Temperature caches:** deleting Celsius clears cached Fahrenheit and Kelvin values.
7. **Positive descriptor:** create a reusable positive-number descriptor with `__delete__`.
8. **Notification invariant:** deleting a notification channel must never leave zero channels configured.

# Summary

A property deleter turns:

```python
del obj.some_property
```

into a controlled operation.

The most important design question is not "how do I delete this attribute?" but:

> **What should deletion mean for this object?**

Once that meaning is clear, the deleter becomes a natural place to encode the rule.